# C. elegans Protein Extraction Pipeline: Molecular Drivers of Embryogenesis

**Author:** Suroush Bastani| **Project:** OpenWorm Studentship

## 1. Introduction & Purpose
The purpose of this notebook is to provide an open-source, automated pipeline for the OpenWorm and DevoWorm communities to extract, organize, and analyze protein sequence data specific to the nematode *Caenorhabditis elegans*. 

This pipeline uses **Biopython** to automate BLAST (Basic Local Alignment Search Tool) searches against the NCBI database, filtering specifically for the *C. elegans* proteome. By automating this process, researchers can easily pull homolog data, sequence alignments, and e-values for key developmental proteins without manual web scraping.

## 2. Literature Context: Expanding on Alicea et al.
In systems biology models of *C. elegans* embryogenesis, such as those explored by **Alicea et al.**,development is tracked through invariant cell lineages and differentiation waves. While these computational models map the structural and topological divisions of the embryo, they must besed on the underlying molecular biology.

The proteins targeted in this pipeline are the physical drivers of the differentiation events modeled by Alicea et al. They include:
* **Wnt Signaling Pathway:** (e.g., APC, AXIN, B-Catenin, GSK3, Wnt, FzR). These proteins dictate cell fate, anterior-posterior polarity, and asymmetric cell division.
* **Cell Cycle Regulators:** (e.g., CDK-1, CDK-2, CDK-4, Cyclin D, Cyclin E, CKI-1). These act as the "clocks" driving the cell divisions mapped in the lineage trees.
* **Morphogenesis & Kinase Cascades:** (e.g., Rho, Rac, Cdc42, ROCK, MAPK, JNK). Essential for cellular shape changes and signal transduction.

By bridging the gap between theoretical lineage models and genomic sequence data, this pipeline provides the raw biological data needed to improve OpenWorm's developmental simulations.

## 3. Setup and Installation
First, we ensure that Biopython is installed and up to date in the environment.

In [1]:
!pip install biopython --upgrade

In [2]:
import json
import zipfile
import os
import time
from Bio.Blast import NCBIWWW
import requests
import re

## 4. Defining the Target Proteins
We define a Python dictionary mapping the protein name to its NCBI accession number. This structure keeps the code modular, allowing researchers to easily add new proteins (such as MELK or other newly annotated homologs) without writing new BLAST functions.

In [3]:
protein_targets = {
    "APC": "SFQ94271.1",
    "AXIN": "AAL77082.1",
    "B-Catenin": "CCE71280.1",
    "CaN": "O02039.1",
    "Cdc14": "NP_001021969.1",
    "Cdc42": "NP_495598.1",
    "CDK-1": "NP_001022747.1",
    "CDK-2": "O61847.2",
    "CDK-4": "AAD48898.1",
    "CK-1": "AAP06180.1",
    "CKI-1": "NP_495641.1",
    "cyclinD": "AAC35273.1",
    "cyclinE": "AAM78547.1",
    "E cad": "CCD72204.1",
    "FzR": "ABA18181.1",
    "GSK3": "CAA22311.1",
    "IP3": "CAB45863.1",
    "JNK": "BAA82640.1",
    "LIN-28": "AAC47476.1",
    "MAPK": "CCD61895.1",
    "MELK": "NP_001023420.2",
    "PLC": "VTW47462.1",
    "PKC": "VTW47465.1",
    "Rac": "CAB04593.1",
    "Rho": "CCD69972.1",
    "ROCK": "P92199.1",
    "Wnt": "AAA64847.1",
}

## 5. The Automation Function
Here we define the function that communicates with the NCBI API. 

**Key Parameters used in `qblast`:**
* `program="blastp"`: Comparing protein sequences to a protein database.
* `database="nr"`: The non-redundant protein database.
* `hitlist_size=250`: Increased from the default 100 to cast a broader net for homologous sequences.
* `expect=0.05`: The E-value threshold to filter out statistically insignificant hits.
* `entrez_query="Caenorhabditis elegans[Organism]"`: The critical filter ensuring we only return data relevant to our target model organism.
* `format_type="JSON2"`: Downloads the results in a modern, easily parseable JSON format rather than XML.

In [4]:
def fetch_and_extract_blast(protein_name, accession_id):
    """
    Fetches BLAST results from NCBI using the requests library directly.
    This bypasses a known Biopython bug with binary ZIP files.
    """
    zip_filename = f"{protein_name}.zip"
    extract_folder = f"{protein_name}_results"
    
    if os.path.exists(extract_folder):
        print(f"[{protein_name}] Already downloaded. Skipping...")
        return
        
    print(f"[{protein_name}] Initiating BLAST for {accession_id}...")
    
    try:
        # 1. Submit the search to NCBI (PUT)
        url = "https://blast.ncbi.nlm.nih.gov/Blast.cgi"
        put_params = {
            "CMD": "Put",
            "PROGRAM": "blastp",
            "DATABASE": "nr",
            "QUERY": accession_id,
            "ENTREZ_QUERY": "Caenorhabditis elegans[Organism]",
            "HITLIST_SIZE": 250,
            "EXPECT": 0.05,
            "GAPCOSTS": "11 1",
            "MATRIX_NAME": "BLOSUM62",
            "WORD_SIZE": 5
        }
        
        res_put = requests.post(url, data=put_params)
        
        # Extract the Request ID (RID) and Estimated Time (RTOE)
        rid_match = re.search(r"RID = (.*)", res_put.text)
        rtoe_match = re.search(r"RTOE = (.*)", res_put.text)
        
        if not rid_match:
            print(f"[{protein_name}] Failed to start search. NCBI might be overloaded.")
            return
            
        rid = rid_match.group(1).strip()
        wait_time = int(rtoe_match.group(1).strip()) if rtoe_match else 10
        
        print(f"[{protein_name}] Search running (RID: {rid}). Waiting ~{wait_time}s...")
        time.sleep(wait_time)
        
        # 2. Check status until ready (GET)
        while True:
            time.sleep(5) # Poll every 5 seconds
            status_res = requests.get(url, params={"CMD": "Get", "FORMAT_OBJECT": "SearchInfo", "RID": rid})
            
            if "Status=WAITING" in status_res.text:
                continue
            elif "Status=FAILED" in status_res.text or "Status=UNKNOWN" in status_res.text:
                print(f"[{protein_name}] Search failed on NCBI servers.")
                return
            elif "Status=READY" in status_res.text:
                if "ThereAreHits=yes" in status_res.text:
                    break
                else:
                    print(f"[{protein_name}] No homologs found.")
                    return
                    
        # 3. Download the actual ZIP file
        print(f"[{protein_name}] Search complete! Downloading JSON data...")
        dl_res = requests.get(url, params={"CMD": "Get", "FORMAT_TYPE": "JSON2", "RID": rid})
        
        # Save the raw binary content
        with open(zip_filename, "wb") as f:
            f.write(dl_res.content)
            
        # Extract the ZIP
        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)
            
        print(f"[{protein_name}] Successfully extracted to /{extract_folder}")
        
    except Exception as e:
        print(f"[{protein_name}] An error occurred: {e}")

## 6. Pipeline Execution
Now we loop through our dictionary of proteins. 


`# Example for testing: protein_targets = dict(list(protein_targets.items())[:3])`

In [5]:
for name, acc_id in protein_targets.items():
    fetch_and_extract_blast(name, acc_id)

[APC] Initiating BLAST for SFQ94271.1...
[APC] Search running (RID: 0VBYM6TJ016). Waiting ~6s...
[APC] Search complete! Downloading JSON data...
[APC] Successfully extracted to /APC_results
[AXIN] Initiating BLAST for AAL77082.1...
[AXIN] Search running (RID: 0VBZPNFH016). Waiting ~3s...
[AXIN] Search complete! Downloading JSON data...
[AXIN] Successfully extracted to /AXIN_results
[B-Catenin] Initiating BLAST for CCE71280.1...
[B-Catenin] Search running (RID: 0VC626X2014). Waiting ~10s...
[B-Catenin] Search complete! Downloading JSON data...
[B-Catenin] Successfully extracted to /B-Catenin_results
[CaN] Initiating BLAST for O02039.1...
[CaN] Search running (RID: 0VC6TRHG014). Waiting ~3s...
[CaN] Search complete! Downloading JSON data...
[CaN] Successfully extracted to /CaN_results
[Cdc14] Initiating BLAST for NP_001021969.1...
[Cdc14] Search running (RID: 0VC89TZT016). Waiting ~2s...
[Cdc14] Search complete! Downloading JSON data...
[Cdc14] Successfully extracted to /Cdc14_results
[C

## 7. Data Extraction & Verification
Once the JSON files are downloaded and extracted, we need to parse them to verify the data.

The JSON structure contains the `BlastOutput2` object, which holds metadata about our search and the actual `hits`. Each hit represents a biological sequence from the database that aligned well with our query. This data can subsequently be fed into modeling software or used to construct molecular alignment trees.

Below is a demonstration function that opens the resulting JSON file for the APC protein and inspects the top homologous hit.

In [6]:
def inspect_top_hit(protein_folder):
    """Reads the extracted JSON and prints the top BLAST hit details."""
    if os.path.exists(protein_folder):
        json_files = [f for f in os.listdir(protein_folder) if f.endswith('.json')]
        
        # The main results are typically housed in the file ending with _1.json
        target_file = next((f for f in json_files if f.endswith('_1.json')), json_files[0])
        file_path = os.path.join(protein_folder, target_file)
        
        with open(file_path, 'r') as f:
            data = json.load(f)
            
        # Navigate the JSON tree to grab the top hit
        try:
            hits = data['BlastOutput2']['report']['results']['search']['hits']
            top_hit = hits[0]
            
            print(f"--- Top Alignment Hit for {protein_folder.split('_')[0]} ---")
            print(f"Hit ID: {top_hit['description'][0]['id']}")
            print(f"Title: {top_hit['description'][0]['title']}")
            print(f"E-value: {top_hit['hsps'][0]['evalue']}")
            print(f"Identity: {top_hit['hsps'][0]['identity']} out of {top_hit['hsps'][0]['align_len']} amino acids")
        except KeyError:
            print("Could not parse the JSON structure. Check the file format.")
    else:
        print(f"Folder '{protein_folder}' not found. Please run the execution block first.")

# Run the verification check on APC
inspect_top_hit("APC_results")

--- Top Alignment Hit for APC ---
Hit ID: ref|NP_001334202.1|
Title: Adenomatous polyposis coli protein [Caenorhabditis elegans]
E-value: 0
Identity: 892 out of 892 amino acids
